In [10]:
import pandas as pd

# Load your CSV
df = pd.read_csv("../data/processed/emptylabel_movie.csv")

# Accuracy: 1 if results == 0, else 0
df["correct"] = (df["results"] == 0).astype(int)

models = ["HRR_300", "HRR_512", "HRR_1024"]

# ── Helper: accuracy pivot ────────────────────────────────────────────────────
def acc_pivot(group_cols):
    return (
        df[df["model"].isin(models)]
        .groupby(group_cols + ["model"])["correct"]
        .mean()
        .mul(100).round(2)
        .reset_index()
        .pivot_table(index=group_cols, columns="model", values="correct")
        .reindex(columns=models)
        .reset_index()
    )

# ── Helper: avg result count pivot ───────────────────────────────────────────
def count_pivot(group_cols):
    return (
        df[df["model"].isin(models)]
        .groupby(group_cols + ["model"])["results"]
        .mean()
        .round(2)
        .reset_index()
        .pivot_table(index=group_cols, columns="model", values="results")
        .reindex(columns=models)
        .reset_index()
        .rename(columns={m: f"{m}_avg_results" for m in models})
    )

# ── Table 1: by (m, n) ───────────────────────────────────────────────────────
acc_mn   = acc_pivot(["m", "n"])
count_mn = count_pivot(["m", "n"])
table_mn = acc_mn.merge(count_mn, on=["m", "n"])

# Interleave: HRR_300, HRR_300_avg_results, HRR_512, ...
cols_mn = ["m", "n"] + [c for m in models for c in (m, f"{m}_avg_results")]
table_mn = table_mn[cols_mn]

# ── Table 2: by m only ───────────────────────────────────────────────────────
acc_m   = acc_pivot(["m"])
count_m = count_pivot(["m"])
table_m = acc_m.merge(count_m, on="m")

# cols_m = ["m"] + [c for m in models for c in (m, f"{m}_avg_results")]
cols_m = ["m"] + [f"{m}_avg_results" for m in models]
table_m = table_m[cols_m]

# ── Display ───────────────────────────────────────────────────────────────────
fmt_mn = {m: "{:.2f}%" for m in models}
fmt_mn.update({f"{m}_avg_results": "{:.2f}" for m in models})

fmt_m = {m: "{:.2f}%" for m in models}
fmt_m.update({f"{m}_avg_results": "{:.2f}" for m in models})

print("=== Accuracy (%) + Avg Result Count by m × n ===")
# display(table_mn.style.format(fmt_mn))

print("\n=== Accuracy (%) + Avg Result Count by m ===")
display(table_m.style.format(fmt_m))

print(table_m.to_latex(index=False, float_format="%.2f"))

=== Accuracy (%) + Avg Result Count by m × n ===

=== Accuracy (%) + Avg Result Count by m ===


model,m,HRR_300_avg_results,HRR_512_avg_results,HRR_1024_avg_results
0,1,0.00,0.00,0.00
1,2,0.00,0.00,0.00
2,3,0.00,0.00,0.00
3,4,0.00,0.00,0.00
4,5,0.00,0.00,0.00
5,6,0.01,0.00,0.00
6,7,0.37,0.00,0.00
7,8,4.04,0.00,0.00
8,9,43.04,0.14,0.00
9,10,20.60,0.66,0.00


\begin{tabular}{rrrr}
\toprule
m & HRR_300_avg_results & HRR_512_avg_results & HRR_1024_avg_results \\
\midrule
1 & 0.00 & 0.00 & 0.00 \\
2 & 0.00 & 0.00 & 0.00 \\
3 & 0.00 & 0.00 & 0.00 \\
4 & 0.00 & 0.00 & 0.00 \\
5 & 0.00 & 0.00 & 0.00 \\
6 & 0.01 & 0.00 & 0.00 \\
7 & 0.37 & 0.00 & 0.00 \\
8 & 4.04 & 0.00 & 0.00 \\
9 & 43.04 & 0.14 & 0.00 \\
10 & 20.60 & 0.66 & 0.00 \\
11 & 7.20 & 0.29 & 0.00 \\
12 & 19.20 & 16.44 & 0.00 \\
13 & 30.83 & 4.43 & 0.01 \\
14 & 61.42 & 2.85 & 0.06 \\
15 & 101.99 & 10.95 & 0.40 \\
\bottomrule
\end{tabular}

